In [ ]:
# ==================== STOCK DIRECTION PREDICTION ====================
# Project: Will the stock go UP or DOWN tomorrow?

import yfinance as yf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# ========================== 1. Get Data ==========================
stock = "AAPL"   # Change this to any stock you like (e.g. TSLA, MSFT, GOOGL)
data = yf.download(stock, period="5y", interval="1d")   # 5 years of data

# ========================== 2. Feature Engineering ==========================
df = data.copy()

# Basic price features
df['Return'] = df['Close'].pct_change()                    # Daily return
df['MA_10'] = df['Close'].rolling(window=10).mean()       # 10-day moving average
df['MA_50'] = df['Close'].rolling(window=50).mean()       # 50-day moving average
df['Volatility'] = df['Return'].rolling(window=20).std()  # 20-day volatility

# Technical indicators (simple)
df['RSI'] = 100 - (100 / (1 + (df['Close'].diff(1).clip(lower=0).rolling(14).mean() / 
                               abs(df['Close'].diff(1).clip(upper=0).rolling(14).mean()))))

df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)   # 1 = Up, 0 = Down

# Drop rows with NaN values
df = df.dropna()

# ========================== 3. Select Features & Target ==========================
features = ['Return', 'MA_10', 'MA_50', 'Volatility', 'RSI']
X = df[features]
y = df['Target']

# ========================== 4. Train-Test Split ==========================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ========================== 5. Train Model ==========================
model = RandomForestClassifier(
    n_estimators=100,      # number of trees
    random_state=42
)

model.fit(X_train, y_train)

# ========================== 6. Evaluate ==========================
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, predictions))

# ========================== 7. Feature Importance ==========================
importance = pd.Series(model.feature_importances_, index=features)
importance.sort_values(ascending=True).plot(kind='barh', title='Feature Importance')
plt.show()

ModuleNotFoundError: No module named 'yfinance'